# 外包数据清洗全流程：从 `lenth9` 到 `full_data`（30 列）

从**已生成的 `lenth9.dta`** 出发，重建第一阶段实证底表 `full_data.dta`，并给出核心描述统计。

## 数据血统

```
[附录] 1718_total_cleaned_by_year1.dta → lenth15 → lenth9      ← 已跑过，不重复生成
 ▼
lenth9.dta                     (465,487,031 行 / 2,778 产品 / 7,191,877 企业)
 │ [Step1]  firm×product×year 聚合；外包额 = min(投入, 产出)
 ▼ firm_product_year_level.dta                (7 列, 90,296,650 行)
 │ [Step2]  产品级特征聚合
 ▼ product_characteristics.dta                (11 列, 2,778 产品)
 │ [Step3]  firm×year 汇总：外包强度、中介/外包标记
 │ [Step4]  主产品 = 自产产值 production_value 最大
 │ [Step5]  合并 similarity + 产品特征（_p 后缀）
 ▼ full_data_rebuild.dta   ← 第一阶段回归底表   (30 列, 90,296,650 行)
 │ [Step6]  与现有 full_data 对比验证
 │ [Step7]  描述统计
```

## 关键口径

- **外包产品**：同一企业同一年对同一产品既买(投入)又卖(产出)。
- **外包额** = `min(投入额, 产出额)`（逐 firm×product×year）——只算"买来又卖掉"的部分，买来自用的归自产。
- **自产额** `production_value` = 产出额 − 外包额。
- **外包强度** = `Σ外包额 / Σ产出额`（firm×year）。
- **中介** `is_intermediary` = 强度 > 0.90；**外包企业** `is_outsourcing` = 强度 ≥ 0.01。
- **主产品** = firm×year 内 **`production_value` 最大**者（并列取 `product_id` 最小，确定性）。

> ⚠️ **与旧版 `full_data.dta` 的差异**：旧版主产品用 `total_output` 最大。本 notebook 已按项目现行定义改为 `production_value` 最大，因此 `is_main` / `main_product` / `main_product_output` / `sales_relative_main` / `input_similarity` / `output_similarity` 六列会与旧版不同——这是**预期的口径修正**，Step 6 的验证会把这几列单独标注。

> ⚠️ **内存**：lenth9 有 4.65 亿行，Step 1 采用分块读入 + 分块聚合，峰值内存可控。需在 VM（大内存）运行。

In [ ]:
import pandas as pd
import numpy as np
import gc, os
from pathlib import Path

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 200)

# ==== 路径约定 ====
#   代码 → Empirical1        （git 同步）
#   数据 → Empirical1_data   （.dta 全放这里，不进 git）
BASE = Path(r'G:\Kuangyu_Temp\Outsource')                          # VM
# BASE = Path(r'C:\Users\HKUBS\Documents\aproject\Outsourcing')    # 本地

CODE = BASE / 'Empirical1'          # 所有代码（ipynb / do）
DATA = BASE / 'Empirical1_data'     # 所有 .dta

# 已生成的大文件（lenth15 / lenth9 / lenth9_clean / lenth9_domin），不重复生成
BIG    = Path(r'G:\Kuangyu_Temp\Data')          # VM
# BIG  = Path(r'C:\Users\HKUBS\Documents\aproject\Data')   # 本地
LENTH9 = BIG / 'lenth9.dta'                     # ★ 本 notebook 的起点

SIM          = DATA / 'full_product_similarity.dta'   # 产品对相似度（输入）
OLD_FULLDATA = DATA / 'full_data.dta'                 # 现有底表（Step 6 对比用）

# 输出（全部落到 DATA）
OUT_FPY      = DATA / 'firm_product_year_level.dta'
OUT_PCHARS   = DATA / 'product_characteristics.dta'
OUT_FULLDATA = DATA / 'full_data_rebuild.dta'         # 不覆盖现有 full_data

DATA.mkdir(exist_ok=True)
os.chdir(DATA)
print('CODE  :', CODE)
print('DATA  :', DATA)
print('LENTH9:', LENTH9)

## Step 1　读入 `lenth9`，聚合到 firm×product×year

`lenth9.dta` 已预先生成（见附录），5 列：`firm_id / product_id / is_output / year / v`，465,487,031 行。

分块读入并在块内先聚合，再做一次全局聚合，避免一次性把 4.65 亿行装进内存。以**产出侧为主表** left-merge 投入侧（只保留有产出记录的 firm×product×year）：

- `outsourcing_value = min(total_input, total_output)`
- `production_value  = total_output − outsourcing_value`
- `outsourcing_percen = outsourcing_value / total_output`

预期：**90,296,650** 行；firm-year **12,339,537**。

In [ ]:
KEYS = ['year', 'firm_id', 'product_id', 'is_output']

parts = []
for i, ch in enumerate(pd.read_stata(LENTH9, chunksize=20_000_000), 1):
    parts.append(ch.groupby(KEYS, as_index=False)['v'].sum())
    print(f'  chunk {i} done')
agg = pd.concat(parts, ignore_index=True).groupby(KEYS, as_index=False)['v'].sum()
del parts; gc.collect()
print('lenth9 聚合后:', f'{len(agg):,}', '行')

out = (agg[agg['is_output'] == 1].drop(columns='is_output')
         .rename(columns={'v': 'total_output'}))
inp = (agg[agg['is_output'] == 0].drop(columns='is_output')
         .rename(columns={'v': 'total_input'}))
del agg; gc.collect()

fpy = out.merge(inp, on=['year', 'firm_id', 'product_id'], how='left')
fpy['total_input'] = fpy['total_input'].fillna(0)
del out, inp; gc.collect()

fpy['outsourcing_value']  = fpy[['total_input', 'total_output']].min(axis=1)
fpy['production_value']   = fpy['total_output'] - fpy['outsourcing_value']
fpy['outsourcing_percen'] = (fpy['outsourcing_value'] / fpy['total_output']).fillna(0)
fpy['year'] = fpy['year'].astype(int)

fpy = fpy[['year', 'firm_id', 'product_id', 'total_output', 'outsourcing_value',
           'production_value', 'outsourcing_percen']]
fpy.to_stata(OUT_FPY, write_index=False)
print('firm_product_year_level 行数:', f'{len(fpy):,}',
      '| firm-year:', fpy.groupby(['firm_id', 'year']).ngroups)

## Step 2　产品级特征 → `product_characteristics.dta`

按 `product_id` 跨 firm、跨年聚合，得到每个产品在全经济体中的特征。`num_firms` / `num_firms_outsourcing` **跨年去重**。

这 10 列（除 merge key）稍后会以 `_p` 后缀并入 `full_data`，用作回归里的市场规模控制变量。

预期：**2,778** 个产品 × 11 列；产品级 outsourcing_intensity 均值 ≈ **0.151**。

In [ ]:
num_firms = (fpy[['product_id', 'firm_id']].drop_duplicates()
             .groupby('product_id', as_index=False).size()
             .rename(columns={'size': 'num_firms'}))

num_firms_os = (fpy[fpy['outsourcing_value'] > 0][['product_id', 'firm_id']]
                .drop_duplicates()
                .groupby('product_id', as_index=False).size()
                .rename(columns={'size': 'num_firms_outsourcing'}))

pchars = fpy.groupby('product_id', as_index=False).agg(
    total_output      = ('total_output',      'sum'),
    total_outsourcing = ('outsourcing_value', 'sum'),
    total_production  = ('production_value',  'sum'),
    num_years         = ('year',              'nunique'),
)
pchars = pchars.merge(num_firms,    on='product_id', how='left')
pchars = pchars.merge(num_firms_os, on='product_id', how='left')
pchars['num_firms_outsourcing'] = pchars['num_firms_outsourcing'].fillna(0).astype(int)

pchars['outsourcing_intensity'] = (pchars['total_outsourcing'] / pchars['total_output']).fillna(0)
pchars['avg_output_per_firm']   = pchars['total_output'] / pchars['num_firms']
pchars['avg_output_per_year']   = pchars['total_output'] / pchars['num_years']
pchars['pct_firms_outsourcing'] = (pchars['num_firms_outsourcing'] / pchars['num_firms']).fillna(0)

pchars = pchars.sort_values('num_firms', ascending=False).reset_index(drop=True)
pchars.to_stata(OUT_PCHARS, write_index=False)
print('product_characteristics:', f'{len(pchars):,}', '产品 x', pchars.shape[1], '列')
print('产品级 outsourcing_intensity 均值:', round(pchars['outsourcing_intensity'].mean(), 4))
del num_firms, num_firms_os; gc.collect()

## Step 3　firm×year 汇总：外包强度、中介/外包标记

- `outsourcing_intensity = firm_total_outsource / firm_total_output`（min 口径）
- `is_intermediary = 强度 > 0.90`，`is_outsourcing = 强度 ≥ 0.01`

对照：新口径下中介占比约 **6.24%**（旧的"整块购入"口径为 12.46%）。

In [ ]:
firm_summary = fpy.groupby(['year', 'firm_id'], as_index=False).agg(
    firm_total_output    = ('total_output',      'sum'),
    firm_total_outsource = ('outsourcing_value', 'sum'),
    n_products           = ('product_id',        'count'),
)
firm_summary['outsourcing_intensity'] = (
    firm_summary['firm_total_outsource'] / firm_summary['firm_total_output']).fillna(0)
firm_summary['is_intermediary'] = (firm_summary['outsourcing_intensity'] > 0.90).astype(int)
firm_summary['is_outsourcing']  = (firm_summary['outsourcing_intensity'] >= 0.01).astype(int)

print('firm-year 观测:', f"{len(firm_summary):,}")
print('中介占比:     {:.2%}'.format(firm_summary['is_intermediary'].mean()))
print('外包企业占比: {:.2%}'.format(firm_summary['is_outsourcing'].mean()))

## Step 4　主产品 = 自产产值 `production_value` 最大

主产品取 firm×year 内 **`production_value` 最大**者（并列取 `product_id` 最小，保证确定性），而不是总产出最大。同时构造：

- `sales_percen        = total_output / firm_total_output`
- `sales_relative_main = total_output / 主产品的 total_output`

> 旧版 `full_data.dta` 用 `total_output` 最大；此处已按项目现行定义修正。

In [ ]:
fpy_sorted = fpy.sort_values(['year', 'firm_id', 'production_value', 'product_id'],
                             ascending=[True, True, False, True])
main = (fpy_sorted.groupby(['year', 'firm_id'], as_index=False)
        .first()[['year', 'firm_id', 'product_id', 'total_output']]
        .rename(columns={'product_id': 'main_product', 'total_output': 'main_product_output'}))

df = fpy.merge(main, on=['year', 'firm_id'], how='left')
df['is_main'] = (df['product_id'] == df['main_product']).astype(int)

df = df.merge(firm_summary, on=['year', 'firm_id'], how='left')
df['sales_percen']        = df['total_output'] / df['firm_total_output']
df['sales_relative_main'] = df['total_output'] / df['main_product_output']
del fpy_sorted, main; gc.collect()
print('主产品并入完成，df 行数:', f'{len(df):,}')

## Step 5　合并 similarity + 产品特征 → `full_data.dta`（30 列）

1. **similarity**：`full_product_similarity.dta` 是产品对（product_1, product_2）的 `input_similarity` / `output_similarity`（来自投入产出表）。对称化后按 (product_id, main_product) 合并；**主产品与自身的相似度定义为 1**。
2. **产品特征**：merge Step 2 的 `product_characteristics`，与 firm-product 级同名的 4 列加 `_p` 后缀（`total_output_p` / `total_outsourcing_p` / `total_production_p` / `outsourcing_intensity_p`）。

最终 **30 列** = 20 列基础 + 10 列产品级特征。

In [ ]:
# --- 5a. similarity ---
sim = pd.read_stata(SIM)
sim1 = sim.rename(columns={'product_1': 'product_id', 'product_2': 'main_product'})
sim2 = sim.rename(columns={'product_2': 'product_id', 'product_1': 'main_product'})
sim_lu = (pd.concat([sim1, sim2], ignore_index=True)
            .drop_duplicates(subset=['product_id', 'main_product']))
del sim, sim1, sim2; gc.collect()

for c in ['product_id', 'main_product']:
    df[c]     = df[c].astype(str).str.strip()
    sim_lu[c] = sim_lu[c].astype(str).str.strip()

df = df.merge(sim_lu[['product_id', 'main_product', 'input_similarity', 'output_similarity']],
              on=['product_id', 'main_product'], how='left')
df.loc[df['is_main'] == 1, ['input_similarity', 'output_similarity']] = 1.0
del sim_lu; gc.collect()

cols20 = ['year', 'firm_id', 'product_id', 'total_output', 'outsourcing_value', 'production_value',
          'outsourcing_percen', 'sales_percen', 'sales_relative_main', 'is_main', 'main_product',
          'main_product_output', 'input_similarity', 'output_similarity', 'firm_total_output',
          'firm_total_outsource', 'n_products', 'outsourcing_intensity', 'is_intermediary', 'is_outsourcing']
df = df[cols20]

# --- 5b. 产品级特征（与 firm-product 级同名的列加 _p 后缀）---
pc = pchars.rename(columns={
    'total_output':          'total_output_p',
    'total_outsourcing':     'total_outsourcing_p',
    'total_production':      'total_production_p',
    'outsourcing_intensity': 'outsourcing_intensity_p',
})
pc['product_id'] = pc['product_id'].astype(str).str.strip()
df = df.merge(pc, on='product_id', how='left')

df = df.sort_values(['year', 'firm_id', 'total_output'],
                    ascending=[True, True, False]).reset_index(drop=True)
df.to_stata(OUT_FULLDATA, write_index=False)
print('full_data.dta:', f'{len(df):,}', '行 ×', df.shape[1], '列')
print('  企业:', df['firm_id'].nunique(), '| 产品:', df['product_id'].nunique())
print('  列名:', list(df.columns))

## Step 6　与现有 `full_data.dta` 对比验证

与 `Empirical1_data/full_data.dta`（现有底表）逐列比对（chunked 读，省内存）。

**预期一致**：行数、列集，以及 `total_output` / `outsourcing_value` / `production_value` / `outsourcing_percen` / `sales_percen` / `firm_total_output` / `firm_total_outsource` / `n_products` / `outsourcing_intensity` / `is_intermediary` / `is_outsourcing` / 10 个 `_p` 列。

**预期不一致（口径修正所致）**：`is_main` / `main_product_output` / `sales_relative_main` / `input_similarity` / `output_similarity`——旧版主产品用 `total_output` 最大，本版改为 `production_value` 最大。

In [ ]:
num_cols = df.select_dtypes('number').columns.tolist()
n_old, old_cols, sums_old = 0, None, None
for ch in pd.read_stata(OLD_FULLDATA, chunksize=5_000_000):
    n_old += len(ch)
    if old_cols is None:
        old_cols = list(ch.columns)
    common = [c for c in num_cols if c in ch.columns]
    s = ch[common].sum()
    sums_old = s if sums_old is None else sums_old + s

print(f'行数:  旧 {n_old:,}   新 {len(df):,}   match = {n_old == len(df)}')
print(f'列数:  旧 {len(old_cols)}   新 {df.shape[1]}   列集相同 = {set(old_cols) == set(df.columns)}')
print(f'仅旧有: {sorted(set(old_cols) - set(df.columns))}')
print(f'仅新有: {sorted(set(df.columns) - set(old_cols))}')

cmp = pd.DataFrame({'old': sums_old, 'new': df[sums_old.index].sum()})
cmp['rel_diff'] = ((cmp['new'] - cmp['old']) / cmp['old'].abs().replace(0, np.nan)).abs()

EXPECTED_DIFF = ['is_main', 'main_product_output', 'sales_relative_main',
                 'input_similarity', 'output_similarity']
cmp['note'] = np.where(cmp.index.isin(EXPECTED_DIFF), '预期不同(主产品口径)', '')

chk = cmp[~cmp.index.isin(EXPECTED_DIFF)]
bad = chk[chk['rel_diff'] > 1e-9]
print('\n【应当一致的列】rel_diff > 1e-9 即异常:')
print(('  异常:\n' + bad.to_string()) if len(bad) else '  全部一致 OK')

print('\n【全列对比】')
print(cmp.round(8).to_string())

## Step 7　描述统计

用当前口径复现第一阶段核心描述性事实。

In [ ]:
# 7.1 企业分类（firm-year）
fy = firm_summary.copy()
fy['ftype'] = np.where(fy['is_intermediary'] == 1, 'Intermediary',
               np.where(fy['outsourcing_intensity'] >= 0.01, 'Outsourcing', 'Pure self'))

tab = fy.groupby('ftype').agg(
    firm_years   = ('firm_id', 'size'),
    unique_firms = ('firm_id', 'nunique'),
    total_output = ('firm_total_output', 'sum'),
).reset_index()
tab['pct_firm_years'] = tab['firm_years'] / tab['firm_years'].sum()
print('=== 7.1 企业分类（firm-year）===')
print(tab.to_string(index=False))

# 7.2 Scope gap：产品对分解（剔除中介）
non_int = df[df['is_intermediary'] != 1]
bk = np.where(non_int['outsourcing_percen'] <= 0, 'Pure self',
      np.where(non_int['outsourcing_percen'] >= 1, 'Pure outsourcing', 'Mixed'))
sg = non_int.groupby(bk).agg(n_pairs=('total_output', 'size'), sales=('total_output', 'sum'))
sg['pct_pairs'] = sg['n_pairs'] / sg['n_pairs'].sum()
sg['pct_sales'] = sg['sales']   / sg['sales'].sum()
print('\n=== 7.2 Scope gap（剔除中介）===')
print(sg[['n_pairs', 'pct_pairs', 'pct_sales']].round(4).to_string())

# 7.3 外包普遍率与强度分布
print('\n=== 7.3 外包普遍率（按年）===')
print(firm_summary.groupby('year')['is_outsourcing'].mean().round(4).to_string())

os_firms = firm_summary[(firm_summary['is_intermediary'] == 0) &
                        (firm_summary['outsourcing_intensity'] > 0)]
print('\n外包强度分布（剔除中介、强度>0）:')
print(os_firms['outsourcing_intensity'].describe(percentiles=[.1, .25, .5, .75, .9, .99]).round(4).to_string())

## 输出文件

全部落在 `Empirical1_data/`：

| 文件 | 内容 |
|---|---|
| `firm_product_year_level.dta` | firm×product×year：total_output / outsourcing_value / production_value / outsourcing_percen（**7 列**，90,296,650 行）|
| `product_characteristics.dta` | 产品级特征（**11 列**，2,778 产品）|
| `full_data_rebuild.dta` | **第一阶段回归底表（30 列）** = 20 列基础 + 10 列产品级特征（`_p` 后缀）|

> 输出用 `full_data_rebuild.dta` 命名，**不覆盖**现有 `full_data.dta`；Step 6c 验证通过后再手动替换。

**口径**：外包额 = min(投入,产出)；主产品 = **production_value 最大**（并列取 product_id 最小）；中介 = 强度 > 0.90；外包企业 = 强度 ≥ 0.01。

**与旧版差异**：旧 `full_data.dta` 主产品用 `total_output` 最大，本版改为 `production_value`。受影响的列：`is_main` / `main_product` / `main_product_output` / `sales_relative_main` / `input_similarity` / `output_similarity`。

---

# 附录　从原始交易数据重建 `lenth15` / `lenth9`

`lenth15.dta` 和 `lenth9.dta` **已在 `G:\Kuangyu_Temp\Data` 生成**，正常流程无需重跑本附录。
以下代码保留以备追溯或原始数据更新时重建。

## A1　原始交易 → lenth15（Stata `database.do`）

原始文件极大（数十亿行），pandas 打不开，collapse 交给 Stata：
`collapse (sum) v, by(firm_id product_id input_output year)`、drop `v<=0`、截 15 位、`is_output`、只保留有产出的企业。

## A2　15 位码 → 9 位码标准化

商品编码分层（1/3/5/7/9 位为层级节点，后面补 0）。统一到 9 位，但有些产品实际只细到 5 或 7 位：

1. 剔除**高层聚合码**（1 位或 3 位后全 0，太粗）。
2. 判断真实层级：某 7 位前缀只对应一个 9 位码 → 该产品最多到 7 位；5 位同理。
3. 合法 9 位码集合 = 真 9 位码（不以 `00` 结尾）+ 7 位层级码（补 `00`）。
4. 用该集合过滤，在 9 位层级重新聚合。
5. 只保留**既产出又投入**的企业。

结果：产品 4058 → **2778**，企业 **7,191,877**，行数 **465,487,031**。

In [ ]:
# A1　原始交易 → lenth15（需要时才跑）
import subprocess

RAW       = Path(r'G:\Kuangyu_Temp\single_product\1718_total_cleaned_by_year1.dta')
stata_exe = r"C:/Program Files/Stata17/StataMP-64.exe"
DO_FILE   = str(CODE / 'pipeline' / 'database.do')

proc = subprocess.run([stata_exe, "/e", "do", DO_FILE],
                      capture_output=True, text=True, errors="ignore")
print('Stata return code:', proc.returncode, '(0 = 正常)')

df15 = pd.read_stata(BIG / 'lenth15.dta')
print('lenth15 行数:', f'{len(df15):,}',
      '| 企业:', df15['firm_id'].nunique(),
      '| 15位产品:', df15['product_id'].nunique())

In [ ]:
# A2　15 位码 → 9 位码（需要时才跑）
df1 = df15
df1['product_id'] = df1['product_id'].astype(str)
df2 = df1[df1['is_output'] == 1].copy()   # 产出侧
df3 = df1[df1['is_output'] == 0].copy()   # 投入侧
df2['product_id_9'] = df2['product_id'].str[:9]
df3['product_id_9'] = df3['product_id'].str[:9]

def is_high_level(code):
    return (code[1:] == '0' * (len(code) - 1)) or (code[3:] == '0' * (len(code) - 3))

df_low = df2[~df2['product_id'].apply(is_high_level)]

g = df_low.groupby('product_id', as_index=False)['v'].sum()
g['p5'] = g['product_id'].str[:5]
g['p7'] = g['product_id'].str[:7]
g['p9'] = g['product_id'].str[:9]

c7 = g.groupby('p7')['p9'].nunique(); single7 = set(c7[c7 == 1].index)
c5 = g.groupby('p5')['p7'].nunique(); single5 = set(c5[c5 == 1].index)

five_in_seven  = [x for x in single7 if x.endswith('00')]
seven_in_seven = [x for x in single7 if not x.endswith('00')]
for i in five_in_seven:
    if i[:-2] in single5:
        seven_in_seven.append(i)
seven_in_seven = [i + '00' for i in seven_in_seven]

p9_true = [x for x in g['p9'].drop_duplicates().tolist() if not x.endswith('00')]
product_id_9_final = set(p9_true + seven_in_seven)
print('最终合法 9 位码数:', len(product_id_9_final))

def to9(dd):
    dd = dd[dd['product_id_9'].isin(product_id_9_final)].copy()
    dd = dd.drop(columns=['product_id']).rename(columns={'product_id_9': 'product_id'})
    return dd.groupby(['firm_id', 'product_id', 'is_output', 'year'], as_index=False)['v'].sum()

out9 = to9(df2)
in9  = to9(df3)

firms = set(out9['firm_id']) & set(in9['firm_id'])   # 既产出又投入的企业
out9 = out9[out9['firm_id'].isin(firms)]
in9  = in9[in9['firm_id'].isin(firms)]

df9 = pd.concat([in9, out9], ignore_index=True)
print('lenth9 行数:', f'{len(df9):,}',
      '| 企业:', df9['firm_id'].nunique(),
      '| 产品:', df9['product_id'].nunique())
df9.to_stata(BIG / 'lenth9.dta', write_index=False)
del df1, df2, df3, df_low, g, out9, in9, df15, df9; gc.collect()